In [15]:

"pip install nltk"

'pip install nltk'

In [16]:
"pip install scikit-learn"

'pip install scikit-learn'

In [17]:
"pip install pandas"

'pip install pandas'

Importting libraries

In [18]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
import re
import nltk
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
import pandas as pd


In [19]:
# Download required NLTK data
nltk.download('punkt')
nltk.download('stopwords')
nltk.download('wordnet')

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\Hp\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping tokenizers\punkt.zip.
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\Hp\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping corpora\stopwords.zip.
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\Hp\AppData\Roaming\nltk_data...


True

Loading and spliting Dataset

In [23]:
# Read the CSV file
path = "./CEAS_08.csv"
df = pd.read_csv(path)

# Split into benign and phishing datasets
benign_df = df[df['label'] == 0][['sender', 'receiver', 'date', 'subject', 'body', 'label', 'urls']]
phishing_df = df[df['label'] == 1][['sender', 'receiver', 'date', 'subject', 'body', 'label', 'urls']]

In [30]:
import nltk

nltk.download('punkt')        # for word_tokenize
nltk.download('stopwords')    # for stopwords
nltk.download('wordnet')      # for WordNetLemmatizer
nltk.download('omw-1.4')      # for lemmatizer's dictionary support


[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\Hp\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\Hp\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\Hp\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to
[nltk_data]     C:\Users\Hp\AppData\Roaming\nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


True

In [45]:
import nltk
nltk.download('punkt')  # standard tokenizer data


[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\Hp\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!


True

In [46]:
def preprocess_text(text):
                       # Remove HTML tags
                       text = text.strip()
    
                       # Convert to lowercase
                       text = text.lower()
    
                       # Remove URLs
                       text = re.sub(r'http\S+|www\S+|https\S+', '', text, flags=re.MULTILINE)
    
                       # Remove email addresses
                       text = re.sub(r'\S+@\S+', '', text)
    
                       # Remove special characters and numbers
                       text = re.sub(r'[^\w\s]', '', text)
                       text = re.sub(r'\d+', '', text)
    
                       # Tokenization
                       tokens = word_tokenize(text)
    
                       # Remove stopwords
                       stop_words = set(stopwords.words('english'))
                       tokens = [token for token in tokens if token not in stop_words]
    
                       # Lemmatization
                       lemmatizer = WordNetLemmatizer()
                       tokens = [lemmatizer.lemmatize(token) for token in tokens]
    
                       # Remove extra whitespace
                       text = ' '.join(tokens).strip()
    
                       return text

# Clean email text for both datasets
benign_df['clean_text'] = benign_df['body'].apply(preprocess_text)
phishing_df['clean_text'] = phishing_df['body'].apply(preprocess_text)

# Create TF-IDF features for benign emails
benign_tfidf_vectorizer = TfidfVectorizer(
                       max_features=5000,
                       min_df=2,
                       max_df=0.95,
                       ngram_range=(1, 2),
                       stop_words='english'
)
benign_tfidf_features = benign_tfidf_vectorizer.fit_transform(benign_df['clean_text'])

# Create TF-IDF features for phishing emails
phishing_tfidf_vectorizer = TfidfVectorizer(
                       max_features=5000,
                       min_df=2,
                       max_df=0.95,
                       ngram_range=(1, 2),
                       stop_words='english'
)
phishing_tfidf_features = phishing_tfidf_vectorizer.fit_transform(phishing_df['clean_text'])

# Convert to DataFrame for easier handling
benign_tfidf_df = pd.DataFrame(benign_tfidf_features.toarray(), 
                                          columns=benign_tfidf_vectorizer.get_feature_names_out())
phishing_tfidf_df = pd.DataFrame(phishing_tfidf_features.toarray(), 
                                          columns=phishing_tfidf_vectorizer.get_feature_names_out())

print("Benign data sample:")
print(benign_df.head())
print("\nPhishing data sample:")
print(phishing_df.head())

# Show nonzero TF-IDF features for the first benign and phishing email
benign_nonzero_features = benign_tfidf_df.loc[0][benign_tfidf_df.loc[0] > 0]
phishing_nonzero_features = phishing_tfidf_df.loc[0][phishing_tfidf_df.loc[0] > 0]
print("\nBenign email TF-IDF features:")
print(benign_nonzero_features)
print("\nPhishing email TF-IDF features:")
print(phishing_nonzero_features)

Benign data sample:
                                 sender  \
3    Michael Parker <ivqrnai@pobox.com>   
8     qydlqcws-iacfym@issues.apache.org   
15           Racing <uqyrmo@sailing.ie>   
18  Aaron Kulkis <cmiqlkx91@hotpop.com>   
19  Aaron Kulkis <cmiqlkx91@hotpop.com>   

                                          receiver  \
3   SpamAssassin Dev <xrh@spamassassin.apache.org>   
8                      xrh@spamassassin.apache.org   
15                     user5@gvc.ceas-challenge.cc   
18                opensuse <wkilxloc@opensuse.org>   
19                opensuse <wkilxloc@opensuse.org>   

                               date  \
3   Tue, 05 Aug 2008 17:31:20 -0600   
8   Tue, 05 Aug 2008 15:31:03 -0800   
15  Wed, 06 Aug 2008 00:31:14 +0100   
18  Tue, 05 Aug 2008 15:50:37 -0500   
19  Tue, 05 Aug 2008 16:31:41 -0500   

                                              subject  \
3   Re: svn commit: r619753 - in /spamassassin/tru...   
8   [Bug 5780] URI processing turns uuencoded s

Advance Pre Processing  

In [47]:
import pandas as pd
import numpy as np
import re
import string
from datetime import datetime
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import TruncatedSVD
import warnings
from sklearn.preprocessing import StandardScaler
warnings.filterwarnings('ignore')

In [48]:
# Download required NLTK data
nltk.download('punkt', quiet=True)
nltk.download('stopwords', quiet=True)
nltk.download('wordnet', quiet=True)
nltk.download('averaged_perceptron_tagger', quiet=True)

True

In [49]:
# Split into benign and phishing datasets
benign_df = df[df['label'] == 0][['sender', 'receiver', 'date', 'subject', 'body', 'label', 'urls']]
phishing_df = df[df['label'] == 1][['sender', 'receiver', 'date', 'subject', 'body', 'label', 'urls']]

print(f"Total samples: {len(df)}")
print(f"Benign samples: {len(benign_df)}")
print(f"Phishing samples: {len(phishing_df)}")
print(f"Class distribution: {df['label'].value_counts().to_dict()}")


Total samples: 39154
Benign samples: 17312
Phishing samples: 21842
Class distribution: {1: 21842, 0: 17312}


In [50]:
class AdvancedEmailPreprocessor:
    def __init__(self):
        self.lemmatizer = WordNetLemmatizer()
        self.stop_words = set(stopwords.words('english'))
        self.scaler = StandardScaler()
        
    def extract_email_features(self, email_address):
        """Extract features from email address"""
        if pd.isna(email_address) or email_address == '':
            return {
                'email_length': 0,
                'has_numbers': 0,
                'special_char_count': 0,
                'domain_length': 0,
                'suspicious_domain': 0
            }
        
        # Clean email address
        email = str(email_address).strip()
        if '<' in email and '>' in email:
            email = re.findall(r'<([^>]+)>', email)[0] if re.findall(r'<([^>]+)>', email) else email
        
        # Extract domain
        domain = email.split('@')[-1] if '@' in email else ''
        
        # Suspicious domain patterns
        suspicious_patterns = ['temp', 'fake', 'spam', 'test', 'random', 'generated']
        
        return {
            'email_length': len(email),
            'has_numbers': int(any(c.isdigit() for c in email)),
            'special_char_count': sum(1 for c in email if c in string.punctuation),
            'domain_length': len(domain),
            'suspicious_domain': int(any(pattern in domain.lower() for pattern in suspicious_patterns))
        }
    
    def extract_date_features(self, date_str):
        """Extract temporal features from date"""
        if pd.isna(date_str):
            return {'hour': 0, 'day_of_week': 0, 'is_weekend': 0}
        
        try:
            # Parse various date formats
            date_patterns = [
                '%a, %d %b %Y %H:%M:%S %z',
                '%a, %d %b %Y %H:%M:%S',
                '%Y-%m-%d %H:%M:%S',
                '%d/%m/%Y %H:%M:%S'
            ]
            
            parsed_date = None
            for pattern in date_patterns:
                try:
                    parsed_date = datetime.strptime(date_str.split(' (')[0], pattern)
                    break
                except:
                    continue
            
            if parsed_date is None:
                return {'hour': 0, 'day_of_week': 0, 'is_weekend': 0}
            
            return {
                'hour': parsed_date.hour,
                'day_of_week': parsed_date.weekday(),
                'is_weekend': int(parsed_date.weekday() >= 5)
            }
        except:
            return {'hour': 0, 'day_of_week': 0, 'is_weekend': 0}
    
    def extract_url_features(self, text):
        """Extract URL-related features"""
        if pd.isna(text):
            text = ''
        
        text = str(text)
        
        # Find URLs
        url_pattern = r'http[s]?://(?:[a-zA-Z]|[0-9]|[$-_@.&+]|[!*\\(\\),]|(?:%[0-9a-fA-F][0-9a-fA-F]))+'
        urls = re.findall(url_pattern, text)
        
        # Suspicious URL patterns
        suspicious_url_keywords = ['click', 'free', 'win', 'prize', 'urgent', 'limited', 'offer', 'deal']
        
        features = {
            'url_count': len(urls),
            'has_urls': int(len(urls) > 0),
            'suspicious_url_count': 0,
            'avg_url_length': 0,
            'has_ip_url': 0,
            'has_shortened_url': 0
        }
        
        if urls:
            features['avg_url_length'] = np.mean([len(url) for url in urls])
            
            for url in urls:
                # Check for IP addresses in URLs
                ip_pattern = r'\b(?:[0-9]{1,3}\.){3}[0-9]{1,3}\b'
                if re.search(ip_pattern, url):
                    features['has_ip_url'] = 1
                
                # Check for URL shorteners
                shorteners = ['bit.ly', 'tinyurl', 't.co', 'goo.gl', 'ow.ly']
                if any(shortener in url for shortener in shorteners):
                    features['has_shortened_url'] = 1
                
                # Check for suspicious keywords
                if any(keyword in url.lower() for keyword in suspicious_url_keywords):
                    features['suspicious_url_count'] += 1
        
        return features
    
    def clean_text(self, text):
        """Advanced text cleaning"""
        if pd.isna(text):
            return ''
        
        text = str(text).lower()
        
        # Remove URLs
        text = re.sub(r'http[s]?://(?:[a-zA-Z]|[0-9]|[$-_@.&+]|[!*\\(\\),]|(?:%[0-9a-fA-F][0-9a-fA-F]))+', '', text)
        
        # Remove email addresses
        text = re.sub(r'\S+@\S+', '', text)
        
        # Remove HTML tags
        text = re.sub(r'<[^>]+>', '', text)
        
        # Remove extra whitespace and special characters
        text = re.sub(r'[^\w\s]', ' ', text)
        text = re.sub(r'\s+', ' ', text)
        
        # Tokenize and lemmatize
        tokens = word_tokenize(text)
        tokens = [self.lemmatizer.lemmatize(token) for token in tokens 
                 if token not in self.stop_words and len(token) > 2]
        
        return ' '.join(tokens)
    
    def extract_text_features(self, text):
        """Extract statistical features from text"""
        if pd.isna(text):
            text = ''
        
        text = str(text)
        
        return {
            'text_length': len(text),
            'word_count': len(text.split()),
            'sentence_count': len(re.split(r'[.!?]+', text)),
            'avg_word_length': np.mean([len(word) for word in text.split()]) if text.split() else 0,
            'uppercase_ratio': sum(1 for c in text if c.isupper()) / len(text) if text else 0,
            'exclamation_count': text.count('!'),
            'question_count': text.count('?'),
            'digit_count': sum(1 for c in text if c.isdigit()),
            'special_char_ratio': sum(1 for c in text if c in string.punctuation) / len(text) if text else 0
        }




In [51]:
def create_comprehensive_features(df):
    """Create comprehensive feature set from email data"""
    
    print("Extracting sender features...")
    sender_features = df['sender'].apply(preprocessor.extract_email_features)
    sender_df = pd.DataFrame(list(sender_features))
    sender_df.columns = [f'sender_{col}' for col in sender_df.columns]
    
    print("Extracting receiver features...")
    receiver_features = df['receiver'].apply(preprocessor.extract_email_features)
    receiver_df = pd.DataFrame(list(receiver_features))
    receiver_df.columns = [f'receiver_{col}' for col in receiver_df.columns]
    
    print("Extracting date features...")
    date_features = df['date'].apply(preprocessor.extract_date_features)
    date_df = pd.DataFrame(list(date_features))
    
    print("Extracting subject text features...")
    subject_text_features = df['subject'].apply(preprocessor.extract_text_features)
    subject_text_df = pd.DataFrame(list(subject_text_features))
    subject_text_df.columns = [f'subject_{col}' for col in subject_text_df.columns]
    
    print("Extracting body text features...")
    body_text_features = df['body'].apply(preprocessor.extract_text_features)
    body_text_df = pd.DataFrame(list(body_text_features))
    body_text_df.columns = [f'body_{col}' for col in body_text_df.columns]
    
    print("Extracting URL features from body...")
    url_features = df['body'].apply(preprocessor.extract_url_features)
    url_df = pd.DataFrame(list(url_features))
    
    print("Cleaning text data...")
    df['subject_clean'] = df['subject'].apply(preprocessor.clean_text)
    df['body_clean'] = df['body'].apply(preprocessor.clean_text)
    df['combined_text'] = df['subject_clean'] + ' ' + df['body_clean']
    
    # Combine all features
    feature_df = pd.concat([
        sender_df,
        receiver_df, 
        date_df,
        subject_text_df,
        body_text_df,
        url_df,
        df[['label', 'subject_clean', 'body_clean', 'combined_text']].reset_index(drop=True)
    ], axis=1)
    
    return feature_df




In [52]:
class Vectorizer:
    def __init__(self):
        self.tfidf_text = TfidfVectorizer(
            max_features=1000,
            ngram_range=(1, 2),
            stop_words='english'
        )
        
        self.char_vectorizer = TfidfVectorizer(
            analyzer='char',
            ngram_range=(2, 3),
            max_features=300
        )
        
        self.svd = TruncatedSVD(n_components=50, random_state=42)
        
    def fit_transform_text_features(self, feature_df):
        """Create text vectorization"""
        
        text_tfidf = self.tfidf_text.fit_transform(feature_df['combined_text'])
        text_tfidf_df = pd.DataFrame(
            text_tfidf.toarray(),
            columns=[f'text_tfidf_{i}' for i in range(text_tfidf.shape[1])]
        )
        
        char_tfidf = self.char_vectorizer.fit_transform(feature_df['combined_text'])
        char_tfidf_df = pd.DataFrame(
            char_tfidf.toarray(),
            columns=[f'char_tfidf_{i}' for i in range(char_tfidf.shape[1])]
        )
        
        svd_features = self.svd.fit_transform(text_tfidf)
        svd_df = pd.DataFrame(
            svd_features,
            columns=[f'svd_component_{i}' for i in range(svd_features.shape[1])]
        )
        
        return {
            'text_tfidf': text_tfidf_df,
            'char_tfidf': char_tfidf_df,
            'svd_features': svd_df
        }
    
    def create_features(self, feature_df):
        """Create features"""
        
        features = pd.DataFrame()
        
        features['avg_sentence_length'] = feature_df['body_word_count'] / feature_df['body_sentence_count'].clip(lower=1)
        features['url_count'] = feature_df['url_count']
        features['special_char_count'] = feature_df['sender_special_char_count'] + feature_df['receiver_special_char_count']
        features['is_business_hours'] = ((feature_df['hour'] >= 9) & (feature_df['hour'] <= 17)).astype(int)
        
        return features



In [53]:
# Initialize preprocessor
preprocessor = AdvancedEmailPreprocessor()

print("Advanced Email Preprocessor initialized successfully!")

# Split into benign and phishing datasets
benign_df = df[df['label'] == 0][['sender', 'receiver', 'date', 'subject', 'body', 'label', 'urls']]
phishing_df = df[df['label'] == 1][['sender', 'receiver', 'date', 'subject', 'body', 'label', 'urls']]

# Create comprehensive features for benign dataset
print("Creating comprehensive feature set for benign emails...")
benign_feature_df = create_comprehensive_features(benign_df)

print(f"Benign feature dataframe shape: {benign_feature_df.shape}")
print(f"Benign feature columns: {len([col for col in benign_feature_df.columns if col not in ['label', 'subject_clean', 'body_clean', 'combined_text']])}")
print("Benign Feature DataFrame sample:", benign_feature_df.head())

# Create comprehensive features for phishing dataset
print("Creating comprehensive feature set for phishing emails...")
phishing_feature_df = create_comprehensive_features(phishing_df)

print(f"Phishing feature dataframe shape: {phishing_feature_df.shape}")
print(f"Phishing feature columns: {len([col for col in phishing_feature_df.columns if col not in ['label', 'subject_clean', 'body_clean', 'combined_text']])}")
print("Phishing Feature DataFrame sample:", phishing_feature_df.head())

# Initialize vectorizer
vectorizer = Vectorizer()

# Process benign features
benign_text_features = vectorizer.fit_transform_text_features(benign_feature_df)
benign_features = vectorizer.create_features(benign_feature_df)

print("\nBenign Dataset:")
print(f"Text TF-IDF shape: {benign_text_features['text_tfidf'].shape}")
print(f"Character TF-IDF shape: {benign_text_features['char_tfidf'].shape}")
print(f"SVD features shape: {benign_text_features['svd_features'].shape}")
print(f"features shape: {benign_features.shape}")

# Process phishing features
phishing_text_features = vectorizer.fit_transform_text_features(phishing_feature_df)
phishing_features = vectorizer.create_features(phishing_feature_df)

print("\nPhishing Dataset:")
print(f"Text TF-IDF shape: {phishing_text_features['text_tfidf'].shape}")
print(f"Character TF-IDF shape: {phishing_text_features['char_tfidf'].shape}")
print(f"SVD features shape: {phishing_text_features['svd_features'].shape}")
print(f"features shape: {phishing_features.shape}")

# Process numerical features for benign dataset
benign_numerical_features = benign_feature_df.select_dtypes(include=[np.number]).drop('label', axis=1)
print("\nScaling benign numerical features...")
benign_scaler = StandardScaler()
benign_numerical_features_scaled = pd.DataFrame(
    benign_scaler.fit_transform(benign_numerical_features),
    columns=[f'scaled_{col}' for col in benign_numerical_features.columns]
)

# Process numerical features for phishing dataset
phishing_numerical_features = phishing_feature_df.select_dtypes(include=[np.number]).drop('label', axis=1)
print("Scaling phishing numerical features...")
phishing_scaler = StandardScaler()
phishing_numerical_features_scaled = pd.DataFrame(
    phishing_scaler.fit_transform(phishing_numerical_features),
    columns=[f'scaled_{col}' for col in phishing_numerical_features.columns]
)

# Create final feature matrix for benign
print("\nCombining all benign features...")
benign_final_features = pd.concat([
    benign_numerical_features_scaled,
    benign_features,
    benign_text_features['text_tfidf'],
    benign_text_features['char_tfidf'],
    benign_text_features['svd_features']
], axis=1)

# Create final feature matrix for phishing
print("Combining all phishing features...")
phishing_final_features = pd.concat([
    phishing_numerical_features_scaled,
    phishing_features,
    phishing_text_features['text_tfidf'],
    phishing_text_features['char_tfidf'],
    phishing_text_features['svd_features']
], axis=1)

# Add labels
benign_final_features['label'] = benign_feature_df['label'].values
phishing_final_features['label'] = phishing_feature_df['label'].values

print(f"\nBenign final feature matrix shape: {benign_final_features.shape}")
print(f"Phishing final feature matrix shape: {phishing_final_features.shape}")

# Display feature summary for both datasets
for dataset_name, final_features in [("Benign", benign_final_features), ("Phishing", phishing_final_features)]:
    feature_categories = {
        'Scaled Numerical': len([col for col in final_features.columns if col.startswith('scaled_')]),
        'Advanced Engineered': len(benign_features.columns if dataset_name == "Benign" else phishing_features.columns),
        'Text TF-IDF': len([col for col in final_features.columns if col.startswith('text_tfidf_')]),
        'Character TF-IDF': len([col for col in final_features.columns if col.startswith('char_tfidf_')]),
        'SVD Components': len([col for col in final_features.columns if col.startswith('svd_component_')])
    }

    print(f"\n{dataset_name} Feature Categories:")
    for category, count in feature_categories.items():
        print(f"{category}: {count} features")

Advanced Email Preprocessor initialized successfully!
Creating comprehensive feature set for benign emails...
Extracting sender features...
Extracting receiver features...
Extracting date features...
Extracting subject text features...
Extracting body text features...
Extracting URL features from body...
Cleaning text data...
Benign feature dataframe shape: (17312, 41)
Benign feature columns: 37
Benign Feature DataFrame sample:    sender_email_length  sender_has_numbers  sender_special_char_count  \
0                   17                   0                          2   
1                   33                   0                          4   
2                   17                   0                          2   
3                   20                   1                          2   
4                   20                   1                          2   

   sender_domain_length  sender_suspicious_domain  receiver_email_length  \
0                     9                         0    

ML model

In [54]:
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix


In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

# Load dataset
df = pd.read_csv('CEAS_08.csv')  # replace with your actual file path

# Combine subject and body as input text
df['text'] = df['subject'].fillna('') + ' ' + df['body'].fillna('')

# Define features and target
X = df['text']
y = df['label']

# Split data into train/test sets (80% train, 20% test)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Vectorize text using TF-IDF
vectorizer = TfidfVectorizer(stop_words='english', max_features=5000)
X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf = vectorizer.transform(X_test)

# Dictionary of models to test
models = {
    "Naive Bayes": MultinomialNB(),
    "Logistic Regression": LogisticRegression(max_iter=1000),
    "Linear SVM": LinearSVC(),
    "Random Forest": RandomForestClassifier(random_state=42)
}

# Training and evaluating each model
for name, model in models.items():
    model.fit(X_train_tfidf, y_train)
    y_pred = model.predict(X_test_tfidf)
    accuracy = accuracy_score(y_test, y_pred)
    print(f"{name} Accuracy: {accuracy:.4f}")


Naive Bayes Accuracy: 0.9839
Logistic Regression Accuracy: 0.9935
Linear SVM Accuracy: 0.9959
Random Forest Accuracy: 0.9959


In [ ]:
# Dictionary of models to test
models = {
    "Naive Bayes": MultinomialNB(),
    "Logistic Regression": LogisticRegression(max_iter=1000),
    "Linear SVM": LinearSVC(),
    "Random Forest": RandomForestClassifier()
}

# Training and evaluating each model
for name, model in models.items():
    model.fit(X_train_tfidf, y_train)
    y_pred = model.predict(X_test_tfidf)
    accuracy = accuracy_score(y_test, y_pred)
    print(f"{name} Accuracy: {accuracy:.4f}")


Naive Bayes Accuracy: 0.9839
Logistic Regression Accuracy: 0.9935
Linear SVM Accuracy: 0.9959
Random Forest Accuracy: 0.9959


In [58]:
# choosing the best model (Linear SVM Accuracy: 0.9959)
best_model = LinearSVC()
best_model.fit(X_train_tfidf, y_train)
y_pred = best_model.predict(X_test_tfidf)

# Detailed evaluation
print("Classification Report:\n", classification_report(y_test, y_pred))
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred))


Classification Report:
               precision    recall  f1-score   support

           0       1.00      1.00      1.00      3490
           1       1.00      1.00      1.00      4341

    accuracy                           1.00      7831
   macro avg       1.00      1.00      1.00      7831
weighted avg       1.00      1.00      1.00      7831

Confusion Matrix:
 [[3474   16]
 [  16 4325]]


In [59]:
import joblib


joblib.dump(best_model, "phishing_model.pkl")


['phishing_model.pkl']